# Colab Worker C: Track C - Learned Watermark Encoder & Controls
This notebook runs on **Colab Account 3** (T4/V100/A100 GPU).
- Trains Dynamic Band-Allocation Watermarking Network ($k=32$ bits, PSNR=40 dB)
- Runs 100-image sanity overfit check
- Trains 3 Fixed-band Controls: Low (0-2), Mid (3-5), High (6-7)
- Trains Main Model Stage 1 (differentiable noise mixture)
- Once Worker A exports `fitted_g.json`: Trains Main Model Stage 2 with Spectral-Transfer Surrogate (Merge 1)

In [ ]:
# 1. GPU Check & Drive Mount
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')
SHARED_DIR = '/content/drive/MyDrive/watermark_internship'
!mkdir -p $SHARED_DIR/checkpoints

In [ ]:
# 2. Clone repo & setup dependencies
!git clone https://github.com/<YOUR_GITHUB_REPO>.git /content/repo
%cd /content/repo
!pip install -q diffusers transformers accelerate compressai kornia lpips pandas tqdm

In [ ]:
# 3. Unpack image splits
!unzip -q /content/repo/data/processed_splits.zip -d /content/repo/data/processed/

In [ ]:
# 4. Step 1: Sanity Overfit Test (100 images, identity noise)
!python -m track_c.train \
    --manifest data/manifests/train_manifest.json \
    --images_dir data/processed/fit \
    --overfit_100 \
    --epochs 10 \
    --save_dir checkpoints/overfit_check

In [ ]:
# 5. Step 2: Train 3 Fixed-band Controls (Low, Mid, High)
!python -m track_c.train --fixed_regime low --epochs 25 --save_dir checkpoints/control_low
!python -m track_c.train --fixed_regime mid --epochs 25 --save_dir checkpoints/control_mid
!python -m track_c.train --fixed_regime high --epochs 25 --save_dir checkpoints/control_high

!cp -r checkpoints/control_* $SHARED_DIR/checkpoints/

In [ ]:
# 6. Step 3: Train Main Model Stage 1 (without surrogate)
!python -m track_c.train --epochs 30 --save_dir checkpoints/main_stage1
!cp -r checkpoints/main_stage1 $SHARED_DIR/checkpoints/

In [ ]:
# 7. Step 4 (MERGE 1): Train Main Model Stage 2 with Surrogate from Worker A
# Ensure Worker A has finished and uploaded fitted_g.json
import os, time
fitted_path = f'{SHARED_DIR}/results/fitted_g.json'
while not os.path.exists(fitted_path):
    print('Waiting for Worker A fitted_g.json...')
    time.sleep(30)

!python -m track_c.train \
    --surrogate_json $SHARED_DIR/results/fitted_g.json \
    --epochs 30 \
    --save_dir checkpoints/main_stage2_surrogate

!cp -r checkpoints/main_stage2_surrogate $SHARED_DIR/checkpoints/
print('✓ Track C training complete! Models saved to Drive.')